# **SmoothGrad, VarGrad и проверка карты на вменяемость**

Практика к модулю [«Атрибуция: градиент как эвристика»](https://open-xai-platform.web.app).

В уроке мы сказали две вещи и обе оставили без кода:

1. карта Vanilla Gradients зашумлена, и лечится это усреднением по зашумлённым копиям;
2. прежде чем верить любой карте, её надо прогнать через sanity check — «это десяток
   строк кода и один вечер».

Здесь мы напишем эти десять строк. К концу тетради у вас будет:

- своя реализация SmoothGrad, SmoothGrad² и VarGrad — без библиотек, по формулам урока;
- численная проверка тождества $\text{VarGrad} = \text{SmoothGrad}^2 - (\text{SmoothGrad})^2$;
- рандомизация весов и ответ на вопрос, зависит ли ваша карта от того, чему сеть обучилась.

Ноутбук считается на CPU за пару минут.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0);   # шум случайный, но воспроизводимый


## Модель и изображение

Берём ту же связку, что и в остальных практиках блока: ResNet-50 с весами ImageNet
и фотографию из открытого репозитория курса.

**Режим `eval()` обязателен.** В train-режиме BatchNorm считает статистику по батчу
из одной картинки, и прогноз перестаёт совпадать с настоящим — а вместе с ним уезжают
и все атрибуции.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

image = Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/hog.jpg').content)).convert('RGB')
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    target = model(x).argmax().item()
print('Класс:', target, categories[target])

## 1. Vanilla Gradients — то, что мы улучшаем

Напомним формулу карты из урока: берём градиент логита класса по входу и сворачиваем
три канала в один, взяв максимум **по модулю**:

$$M_{ij} = \max_k |w_{ijk}|, \qquad w = \frac{\partial F_c(x)}{\partial x}$$

Порядок операций важен: сначала модуль, потом максимум.

In [ ]:
def vanilla_gradient(model, x, target):
    """Градиент логита класса по входу — тензор формы входа."""
    x = x.clone().requires_grad_(True)
    logit = model(x)[0, target]
    grad, = torch.autograd.grad(logit, x)
    return grad

def to_map(grad):
    """Свёртка каналов в карту: сначала модуль, потом максимум."""
    return grad.abs().max(dim=1).values[0]

vanilla = to_map(vanilla_gradient(model, x, target))
print('размер карты:', tuple(vanilla.shape))

**Задание 1.** Сравните два порядка операций на этой же карте: `grad.abs().max(1)`
и `grad.max(1).values.abs()`. У скольких пикселей из 224×224 значения разойдутся
больше чем на 0.001?

In [ ]:
# Ваш код здесь

## 2. SmoothGrad и его семейство

Все три величины считаются из одного набора карт по зашумлённым копиям:

$$\text{SmoothGrad} = \frac1n\sum_i M(x+\varepsilon_i), \quad
\text{SmoothGrad}^2 = \frac1n\sum_i M(x+\varepsilon_i)^2, \quad
\text{VarGrad} = \text{SmoothGrad}^2 - (\text{SmoothGrad})^2$$

Уровень шума задаётся долей от размаха значений входа: $\sigma = \text{доля}\times(x_{max}-x_{min})$.

In [ ]:
def noisy_maps(model, x, target, sigma_frac=0.15, n=25):
    """n карт по зашумлённым копиям входа. Возвращает тензор (n, H, W)."""
    sigma = sigma_frac * (x.max() - x.min())
    maps = []
    for _ in range(n):
        noisy = x + torch.randn_like(x) * sigma
        maps.append(to_map(vanilla_gradient(model, noisy, target)))
    return torch.stack(maps)

maps = noisy_maps(model, x, target)
smoothgrad = maps.mean(0)
smoothgrad_sq = (maps ** 2).mean(0)
vargrad = smoothgrad_sq - smoothgrad ** 2

print('SmoothGrad  среднее:', round(smoothgrad.mean().item(), 5))
print('SmoothGrad² среднее:', round(smoothgrad_sq.mean().item(), 5))
print('VarGrad     среднее:', round(vargrad.mean().item(), 5))

**Задание 2.** В уроке сказано, что путаница между «средним квадратов» и «квадратом
среднего» — самая частая ошибка реализации. Посчитайте обе величины и убедитесь, что
$\text{SmoothGrad}^2 \geq (\text{SmoothGrad})^2$ поточечно. Сколько пикселей нарушают
это неравенство? (Правильный ответ — ноль: это неравенство Йенсена.)

In [ ]:
# Ваш код здесь

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (m, t) in zip(ax, [(vanilla, 'Vanilla'), (smoothgrad, 'SmoothGrad'),
                          (smoothgrad_sq, 'SmoothGrad²'), (vargrad, 'VarGrad')]):
    a.imshow(m.detach().numpy(), cmap='hot')
    a.set_title(t)
    a.axis('off')
plt.tight_layout()
plt.show()

## 3. Гиперпараметр, который выбираете вы

Уровень шума $\sigma$ — не деталь реализации, а выбор автора объяснения: он меняет карту
в широком диапазоне, и диапазон свой у каждой пары «изображение — модель».

In [ ]:
for frac in (0.05, 0.15, 0.35):
    m = noisy_maps(model, x, target, sigma_frac=frac, n=15).mean(0)
    # доля «массы» карты, попавшая в 5 % самых ярких пикселей
    top = torch.topk(m.flatten(), k=m.numel() // 20).values.sum() / m.sum()
    print(f'σ = {frac:.2f} от размаха → в 5 % ярчайших пикселей {top:.1%} массы карты')

**Задание 3.** Что происходит с концентрацией карты при росте шума и почему?
Ответ запишите словами в две строки.

## 4. Sanity check: зависит ли карта от того, чему сеть обучилась

Проверка из работы [Adebayo et al., 2018](https://arxiv.org/abs/1810.03292): случайно
переинициализируем веса — сначала последнего блока, потом всё глубже — и каждый раз
строим карту заново. Если карта не меняется, она объясняет не модель.

Меру сходства берём простую: корреляция Спирмена между картами, вытянутыми в вектор.

In [ ]:
from scipy.stats import spearmanr

def spearman(a, b):
    return spearmanr(a.flatten().detach().numpy(), b.flatten().detach().numpy()).statistic

def randomize(model, blocks, seed=0):
    """Копия сети со случайно переинициализированными весами перечисленных блоков."""
    import copy
    torch.manual_seed(seed)   # веса случайные, но одни и те же при каждом запуске
    broken = copy.deepcopy(model)
    for name in blocks:
        for m in getattr(broken, name).modules():
            if hasattr(m, 'reset_parameters'):
                m.reset_parameters()
    return broken.eval()

cascade = [['fc'], ['fc', 'layer4'], ['fc', 'layer4', 'layer3'], ['fc', 'layer4', 'layer3', 'layer2']]
print(f'{"переинициализировано":42s} корреляция с исходной картой')
for blocks in cascade:
    broken = randomize(model, blocks)
    m = to_map(vanilla_gradient(broken, x, target))
    print(f'{", ".join(blocks):42s} {spearman(vanilla, m):+.3f}')

**Задание 4.** Корреляция должна падать по мере того, как ломается всё больше слоёв.
Чему равна корреляция после переинициализации `fc` и `layer4` (округлите до сотых)?

**Задание 5.** Проделайте то же самое для карты SmoothGrad (достаточно `n=10`).
Проходит ли SmoothGrad проверку так же, как Vanilla Gradients?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **SmoothGrad, SmoothGrad² и VarGrad считаются из одного набора карт** — разница только
  в том, что берут от набора: среднее, среднее квадратов или дисперсию.
- **VarGrad и SmoothGrad² неотрицательны по построению**, поэтому знак вклада теряется.
  Если вам важно «за класс или против» — берите обычный SmoothGrad.
- **Уровень шума — ваш выбор**, и он меняет карту заметно. Указывайте его рядом с картинкой.
- **Sanity check стоит десяти строк.** Прогоняйте его на своей задаче до того, как строить
  выводы: карта, которая не меняется вслед за весами, объясняет не модель.

Подробный разговор об оценке объяснений — в модуле про оценку качества объяснений.